# Notebook 00 — Hello Graph

The smallest possible thread through the stack: connect to the local Neo4j, insert one `:DiagnosticCode`, query it back, print the result, clean up.

**Before running this notebook**, make sure Neo4j is up:

```bash
make up
```

You should see the Neo4j Browser at <http://localhost:7474> (user: `neo4j`, password from your `.env`).

This notebook is intentionally trivial — it exists to prove the dev environment is wired correctly. Subsequent notebooks (01, 02, …) build on top of it.

## 1. Imports + connection

`va_agent.config` loads `.env` automatically and exposes a `Neo4jSettings` dataclass.

In [1]:
from neo4j import GraphDatabase

from va_agent.config import neo4j_settings

settings = neo4j_settings()
print(f"Connecting to {settings.uri} as {settings.user!r} (database: {settings.database})")

driver = GraphDatabase.driver(settings.uri, auth=(settings.user, settings.password))
driver.verify_connectivity()
print("✓ Connected.")

Connecting to bolt://localhost:7687 as 'neo4j' (database: neo4j)
✓ Connected.


## 2. Insert one Diagnostic Code

We `MERGE` so re-running the notebook is idempotent. The node carries two labels:

- `:CFR` — the namespace label for everything sourced from the Code of Federal Regulations
- `:DiagnosticCode` — the type label

DC 5260 (*Leg, limitation of flexion of*) is the canonical example we'll keep using throughout the project.

In [2]:
with driver.session(database=settings.database) as session:
    session.run(
        """
        MERGE (d:CFR:DiagnosticCode {code: $code})
        SET   d.title       = $title,
              d.body_system = $body_system
        """,
        code="5260",
        title="Leg, limitation of flexion of",
        body_system="musculoskeletal",
    )
print("✓ Inserted DC 5260.")

✓ Inserted DC 5260.


## 3. Query it back

In [3]:
with driver.session(database=settings.database) as session:
    result = session.run(
        "MATCH (d:DiagnosticCode {code: '5260'}) RETURN d.code AS code, d.title AS title, d.body_system AS body_system"
    )
    for row in result:
        print(f"DC {row['code']}: {row['title']}  [{row['body_system']}]")

DC 5260: Leg, limitation of flexion of  [musculoskeletal]


## 4. See it in the Neo4j Browser

Open <http://localhost:7474>, log in, and run:

```cypher
MATCH (d:DiagnosticCode) RETURN d
```

You should see one circle. Click it to inspect its properties.

## 5. Cleanup

Remove the demo node and close the driver.

In [4]:
with driver.session(database=settings.database) as session:
    session.run("MATCH (d:DiagnosticCode {code: '5260'}) DETACH DELETE d")
driver.close()
print("✓ Cleaned up.")

✓ Cleaned up.
